In [158]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 32
max_iters = 1000
eval_interval = 2500
learning_rate = 1e-4
eval_iters = 250

cuda


In [159]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [160]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: '' .join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])


In [161]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[ 0, 44, 61, 58,  1, 47, 62, 79],
        [73,  1, 54, 73,  1, 61, 62, 72],
        [57,  1, 73, 61, 58,  1, 47, 62],
        [73,  1, 65, 54, 72, 73,  9,  1],
        [73, 61, 58, 78,  0, 56, 54, 66],
        [ 1, 25, 73, 61, 65, 58, 73, 62],
        [ 1, 73, 62, 66, 58,  1, 33,  1],
        [ 1, 72, 73, 62, 65, 65,  1, 66],
        [72, 73, 71, 54, 67, 60, 58,  1],
        [ 1, 68, 67, 65, 78,  1, 54,  1],
        [72, 68,  1, 68, 74, 71,  1, 59],
        [54, 64, 58,  1, 62, 73,  1, 54],
        [35, 54, 67, 72, 54, 72,  1, 76],
        [58, 67, 73,  1, 73, 61, 58,  1],
        [ 3, 33,  1, 57, 68,  1, 67, 68],
        [ 3, 39, 59,  1, 56, 68, 74, 71],
        [68, 69, 65, 58,  1, 68, 59,  1],
        [61, 58,  1, 42, 74, 65, 58, 71],
        [73, 58, 71, 71, 62, 55, 65, 58],
        [58, 67,  1, 60, 71, 54, 72, 72],
        [ 1, 60, 68, 62, 67, 60,  1, 73],
        [ 1,  1,  1,  1,  1,  1,  1,  1],
        [ 1, 72, 73, 54, 71, 73,  1, 54],
        [ 0, 73, 61, 58, 6

In [162]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in {'train', 'val'}:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [163]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=-1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


,DCdaEsD;)onw1a xoaR7:)(5"Xx]D?2L1"tYOQh?!TBC:mb-DdO]2Iwxo5j68jy
zP]SDfgpATHoMnZR]F5"-D]csf&D)EXLubPcwQewO]9M1Jn_"F:QizxlWn2r,KiTLuimn2d*&TgZp*i)Q
!5mka9zBCXGhqJEk'iy*]u."37c1PXcA,G6pUyG2Tz27zQRjS0.6"N'pA,*5[wOl2Kr_d6Pc'UIBtWv4)2V.iaRynb82?r1CrTie
K!M)-DB"d 5(tP8sZ:Q-9JG0xlo"4PIYwQb7xE
9xxKBPInF:v.x[sGQGPvpSWn2FK?k"fCy_-."Z:&ISv)j3xgGBrD*]."s;05ncr"IM]N kER-YzHNvD8FR7Ho'lDnFCsj3TLQW3u4!y*oGj,d(7&t)-Mkx-!Z8Tm5W"LdRV1?E)*7mEsn_"1J;7]t)Q7!5:nm?iRMo'[wx3CttDD_V3b;0*4fBIcrHoa'tbCKnCDCv_CXMn[JzWY(sbBL


In [164]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.3f}, val loss {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step 0: train loss 4.966, val loss 4.962
4.83546257019043


In [165]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


kZ]qVipjHt,Od
]Onu
r,nzvi65:oYxOL1*yn_"GBCXAA&hfNt)03fqbub!B)]Pe8hg?y0.ieBNYYB5sgQzYSEND?mauZRjic"j333qAHJ,D)U(
"PTEKzvnzpdRMFLpAia]I:icw_Ht5v)*sD8(dTZ2F(ZqSENLpo'[z5se-D4fwDA"2(ioikjdTrD;plL1!9ENYVaYVwgCXi"YiCdJbLbrKr_jR64YdBksh,pxhee
NI;RnU7x]?!!&IpJbG,OAt-D!a8[Ad
mmEBrxXvDBJ
v-g;my;lLoIxF!vT[wnC1Jr,Q7qA])3jj3T[
"Fy7V9Quz1m"4!zBkslYE6]9.dwn-DO_9YR74!M fW"FHoq3TZW-j36mMhN-y90KWVGh3[ UJ'?mNz'8TEqQA3S!bkCc6goV(wn2KB3XfCQy v-)(eBeBL2GLK;1&sDcw;FUeJ[8])bkcrD__GsIdR lQ(1DDTeJ5,Kb
myOscom0ivuz2(Rj)l6
